# 09 — Módulos, archivos y manejo de errores

## Imports

Python resuelve módulos en este orden: built-ins → módulos instalados → módulos locales.

In [1]:
# Importar módulo completo
import math
print(math.sqrt(144))   # 12.0

# Importar nombres específicos
from math import sqrt, log, pi
print(sqrt(144), round(pi, 5))

# Alias — útil para nombres largos o convenciones (pd, np, plt)
import json as js
print(js.dumps({'clave': 42}))

# from ... import * es desaconsejable — contamina el namespace
# from math import *  # evitar


12.0
12.0 3.14159
{"clave": 42}


## pathlib — manejo de rutas

`pathlib.Path` reemplaza `os.path`. Las rutas son objetos, no strings.

In [2]:
from pathlib import Path

# Path() acepta strings y usa / para concatenar (funciona en Windows y Linux)
ruta = Path('datos') / 'ventas' / '2024.csv'
print(ruta)           # datos/ventas/2024.csv
print(ruta.suffix)    # .csv
print(ruta.stem)      # 2024
print(ruta.parent)    # datos/ventas
print(ruta.name)      # 2024.csv

# Directorio actual y home
print(Path.cwd())
print(Path.home())

# glob — encontrar archivos por patrón (lazy, devuelve generator)
# csvs = list(Path('datos').glob('**/*.csv'))   # recursivo con **
# csvs = list(Path('datos').glob('*.csv'))      # solo en el directorio


datos\ventas\2024.csv
.csv
2024
datos\ventas
2024.csv
C:\Users\alefe\OneDrive\Documentos\GitHub\data-analytics-project\notebooks\ciencia_de_datos\python
C:\Users\alefe


## datetime

In [3]:
from datetime import date, datetime, timedelta

hoy     = date.today()
ahora   = datetime.now()

print(hoy)
print(ahora.strftime('%d/%m/%Y %H:%M'))   # formatear

# Parsear string a datetime
fecha_str = '2024-11-28'
fecha     = datetime.strptime(fecha_str, '%Y-%m-%d')
print(fecha.year, fecha.month, fecha.day)

# Aritmética de fechas
en_30_dias = hoy + timedelta(days=30)
print(f'En 30 días: {en_30_dias}')

# Diferencia entre fechas devuelve timedelta
inicio  = date(2024, 1, 1)
dias_transcurridos = (hoy - inicio).days
print(f'Días desde el 1 de enero: {dias_transcurridos}')


2026-08-24
24/08/2026 21:08
2024 11 28
En 30 días: 2026-09-23
Días desde el 1 de enero: 966


## try / except / else / finally

Estructura completa del manejo de excepciones. `else` corre si no hubo excepción. `finally` siempre corre.

In [4]:
def parsear_precio(texto: str) -> float | None:
    try:
        return float(texto.strip().replace(',', '.'))
    except ValueError:
        return None
    except AttributeError:
        # texto no es string (es None, int, etc.)
        return None

for entrada in ['15.99', ' 23,50 ', 'N/A', None, 42]:
    print(f'{str(entrada):<12} → {parsear_precio(entrada)}')


# finally — garantiza limpieza independientemente del resultado
def leer_configuracion(ruta: str) -> dict:
    archivo = None
    try:
        archivo = open(ruta, encoding='utf-8')
        return js.load(archivo)
    except FileNotFoundError:
        return {}
    except js.JSONDecodeError as e:
        raise ValueError(f'Configuración inválida: {e}') from e
    finally:
        if archivo:
            archivo.close()   # siempre cierra el archivo


15.99        → 15.99
 23,50       → 23.5
N/A          → None
None         → None
42           → None


## Context managers — with

`with` garantiza que el recurso se libere aunque haya una excepción. El equivalente a `try/finally` pero más limpio.

In [5]:
import json
from pathlib import Path

datos = {'empresa': 'RetailPro', 'año': 2024, 'ventas': 3_250_000}

# Escribir un archivo JSON
ruta = Path('temporal_config.json')
with open(ruta, 'w', encoding='utf-8') as f:
    json.dump(datos, f, ensure_ascii=False, indent=2)

# Leer
with open(ruta, encoding='utf-8') as f:
    cargado = json.load(f)

print(cargado)

# Limpiar el temporal
ruta.unlink()

# Múltiples context managers en una línea (Python 3.10+)
# with open('a.txt') as a, open('b.txt') as b:
#     ...


{'empresa': 'RetailPro', 'año': 2024, 'ventas': 3250000}


## Excepciones personalizadas

In [6]:
class ErrorValidacion(ValueError):
    """Se lanza cuando un valor no pasa las reglas de negocio."""
    def __init__(self, campo: str, valor, razon: str):
        self.campo  = campo
        self.valor  = valor
        self.razon  = razon
        super().__init__(f'[{campo}={valor!r}] {razon}')


def validar_precio(precio: float) -> float:
    if not isinstance(precio, (int, float)):
        raise ErrorValidacion('precio', precio, 'debe ser numérico')
    if precio <= 0:
        raise ErrorValidacion('precio', precio, 'debe ser positivo')
    return float(precio)


for v in [99.99, -5, 'gratis', 0]:
    try:
        print(validar_precio(v))
    except ErrorValidacion as e:
        print(f'Error: {e}')


99.99
Error: [precio=-5] debe ser positivo
Error: [precio='gratis'] debe ser numérico
Error: [precio=0] debe ser positivo


---
## Resumen

| Herramienta | Uso |
|-------------|-----|
| `pathlib.Path` | rutas multiplataforma |
| `Path.glob('**/*.csv')` | buscar archivos |
| `datetime.strptime` | parsear fechas desde string |
| `try/except/else/finally` | manejo completo de errores |
| `with open(...)` | archivos y recursos |
| clase `(Exception)` | excepciones personalizadas |
